### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_tid_11",
    dataset_year="2015",
    domain_str="chemistry & material science",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/3050",
    download_description="""
We download the dataset from OpenML to a predefined folder.

mkdir -p local-data-warehouse/qsar_tid_11/ && wget -P local-data-warehouse/qsar_tid_11/ https://www.openml.org/data/download/1674848/11.arff
""",
    # References
    academic_reference_bibtex="""@article{olier2018meta,
  title={Meta-QSAR: a large-scale application of meta-learning to drug design and discovery},
  author={Olier, Ivan and Sadawi, Noureddin and Bickerton, G Richard and Vanschoren, Joaquin and Grosan, Crina and Soldatova, Larisa and King, Ross D},
  journal={Machine Learning},
  volume={107},
  pages={285--311},
  year={2018},
  publisher={Springer}
}
""",
    academic_reference_bibtex_key="olier2018meta",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We drop the "MOLECULE_CHEMBL_ID" column.
- We shuffle the data as it has an original distribution shift.
- Only one row has a negative value (-6.24) and is thus extremely out-of-distribution, which is likely results from the target being log scaled in the original domain. We remove this row as it is unclear if this is a scaling or labeling error, but we judge it is in all cases a data artifact.
- We do not log scale the target (it could make sense for this distribution), as the original label is already log transformed.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="MEDIAN_PXC50",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import arff


with open(dataset_mold.path/"11.arff") as f:
    data = arff.load(f)

df = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])
df.drop(columns=["MOLECULE_CHEMBL_ID"], inplace=True)

# Remove one negative outlier row
df = df[df["MEDIAN_PXC50"] >= 0].reset_index(drop=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()